# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`, referencing the Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata and print dataset name and description
print("Dataset Title: ", dataset.metadata.name)
print("Description: ", dataset.metadata.description)


## 2. Data Overview
Review available record sets, fields, and their IDs. All entities (record sets, fields, columns) are referenced by their `@id` field.

In [ ]:
# List all record sets and their IDs
record_sets = dataset.record_sets()
print("Available record sets and their @ids:")
for rs in record_sets:
    print(f"- @id: {rs.id} | Name: {rs.name}")
    fields = rs.fields()
    print("  Fields:")
    for field in fields:
        print(f"    - @id: {field.id} | Name: {field.name} | DataType: {field.data_type}")


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all record sets into DataFrames.
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets()]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    # Each record is a dict mapping field @id to value
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Record set @id: {rs_id}")
    print("  Columns (field @ids):", df.columns.tolist())
    print(df.head(), "\n")

# Example: Display columns for the first record set
if len(record_set_ids) > 0:
    main_rs_id = record_set_ids[0]
    print(f"First record set columns: {dataframes[main_rs_id].columns.tolist()}")
    dataframes[main_rs_id].head()


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. All fields and columns are referenced by their `@id`.

In [ ]:
# For demonstration, select main record set and inspect numeric and grouping fields
main_rs_id = record_set_ids[0] if len(record_set_ids) > 0 else None
df = dataframes[main_rs_id] if main_rs_id else pd.DataFrame()

if not df.empty:
    # Find numeric fields by their @id
    numeric_ids = []
    group_ids = []
    fields = dataset.record_set(main_rs_id).fields()
    for field in fields:
        if field.data_type in ["schema:Integer", "schema:Float"]:
            numeric_ids.append(field.id)
        elif field.data_type == "schema:Text":
            group_ids.append(field.id)

    print("Numeric field @ids:", numeric_ids)
    print("Grouping field @ids:", group_ids)

    # Example: Use first numeric field for filtering
    if numeric_ids:
        numeric_field_id = numeric_ids[0]
        threshold = 10
        # Only filter if numeric_field_id exists and is numeric
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize (z-score) the filtered numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by a text/categorical field (if exists)
        group_field_id = group_ids[0] if group_ids else None
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())


## 5. Visualization
Visualize distributions or relationships between fields referenced by their `@id` using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Visualize numeric field distribution if available
if not df.empty and numeric_ids:
    numeric_field_id = numeric_ids[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of field {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field exists, plot mean by group
    group_field_id = group_ids[0] if group_ids else None
    if group_field_id:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        group_means.plot(kind="bar", figsize=(8,4))
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id`. We loaded metadata, extracted record sets and fields, filtered and normalized a numeric field, and visualized data distributions. This approach supports reproducible and standard-compliant data exploration. Future analysis could focus on deeper modeling or custom aggregation using the verified field @ids.